In [1]:
import os
import pandas as pd
import numpy as np
from scipy.stats import shapiro, ttest_ind, mannwhitneyu, f_oneway, levene, kruskal
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from itertools import combinations
from datetime import datetime

In [12]:
a=np.random.random((10,3))
b=np.array([np.mean(a[0,i]) for i in range(3)])
print(b)

[0.76336544 0.26335784 0.96327358]


In [3]:
df = pd.read_csv(os.path.join('./CSV/IQC_Linha_metrics_results_2025-08-01_21-36-06.csv'))

In [4]:
df

,Scores,F1_Scores,Negativity_Class_0,Negativity_Class_1,Negativity_Class_2
0,0.733333,0.738095,0.359755,0.291944,0.362246
1,0.933333,0.932660,0.347276,0.286050,0.361474
2,0.800000,0.791919,0.365563,0.293839,0.362735
3,0.866667,0.866667,0.360088,0.299976,0.363741
4,0.733333,0.733333,0.356874,0.285448,0.362029
5,1.000000,1.000000,0.362403,0.290010,0.362268
6,0.866667,0.865993,0.342438,0.299470,0.362514
7,0.933333,0.932660,0.358825,0.292298,0.362026
8,1.000000,1.000000,0.358381,0.296527,0.364518
9,0.733333,0.740741,0.356916,0.293088,0.363132


In [15]:
def load_and_merge_models(folder_path, model_prefixes, target_columns):
    """Carrega e aglutina arquivos CSV de múltiplos modelos."""
    model_data = {model: [] for model in model_prefixes}
    
    for file in os.listdir(folder_path):
        if file.endswith('.csv'):
            for model in model_prefixes:
                if file.startswith(model):
                    df = pd.read_csv(os.path.join(folder_path, file))
                    available_cols = [col for col in target_columns if col in df.columns]
                    model_data[model].append(df[available_cols])
                    break
    
    return {model: pd.concat(data, ignore_index=True) for model, data in model_data.items()}

def check_normality_and_homogeneity(data_dict, target_column, alpha=0.05):
    """Verifica normalidade (Shapiro-Wilk) e homocedasticidade (Levene)."""
    normality_results = {}
    all_normal = True
    data_to_compare = []
    
    for model, df in data_dict.items():
        if target_column in df.columns:
            data = df[target_column].dropna()
            stat, p = shapiro(data)
            normality_results[model] = {
                'p_value': p,
                'is_normal': p > alpha,
                'data': data
            }
            all_normal &= (p > alpha)
            data_to_compare.append(data)
        else:
            print(f"Aviso: Coluna '{target_column}' não encontrada no modelo {model}")
            normality_results[model] = {
                'p_value': np.nan,
                'is_normal': False,
                'data': None
            }
            all_normal = False
    
    # Verifica homocedasticidade apenas se todos forem normais
    homogeneous = False
    if all_normal and len(data_to_compare) >= 2:
        _, p_levene = levene(*data_to_compare)
        homogeneous = p_levene > alpha
    
    return normality_results, all_normal, homogeneous

def compare_models(data_dict, target_column, alpha=0.05):
    """Compara modelos com testes paramétricos (ANOVA/t-test) ou não paramétricos."""
    # Verifica suposições
    normality_results, all_normal, homogeneous = check_normality_and_homogeneity(
        data_dict, target_column, alpha
    )
    
    # Imprime resultados
    print("\nTeste de normalidade (Shapiro-Wilk):")
    for model, result in normality_results.items():
        print(f"{model}: p = {result['p_value']:.4f} | {'Normal' if result['is_normal'] else 'Não normal'}")
    
    if all_normal:
        print(f"\nTeste de homocedasticidade (Levene): {'Homogêneo' if homogeneous else 'Heterogêneo'}")
    
    # Prepara dados para comparação
    models = [k for k, v in normality_results.items() if v['data'] is not None]
    data_to_compare = [normality_results[model]['data'] for model in models]
    
    # Seleciona teste estatístico
    if len(models) == 2:
        if all_normal:
            stat, p = ttest_ind(*data_to_compare, equal_var=homogeneous)
            test_name = "Teste t de Student (Welch)" if not homogeneous else "Teste t de Student"
        else:
            stat, p = mannwhitneyu(*data_to_compare)
            test_name = "Teste de Mann-Whitney U"
    else:
        if all_normal:
            stat, p = f_oneway(*data_to_compare)
            test_name = "ANOVA unidirecional"
        else:
            stat, p = kruskal(*data_to_compare)
            test_name = "Teste de Kruskal-Wallis"
    
    # Executa e mostra resultados
    print(f"\n{test_name}:")
    print(f"Estatística = {stat:.4f}, p-value = {p:.4f}")
    
    if p < alpha:
        print(f"Conclusão (α={alpha}): Há diferenças significativas entre os modelos.")
        
        # Testes post-hoc
        if len(models) > 2:
            print("\nTestes post-hoc:")
            
            # Para ANOVA (Tukey HSD)
            if all_normal:
                tukey = pairwise_tukeyhsd(
                    endog=np.concatenate(data_to_compare),
                    groups=np.repeat(models, [len(x) for x in data_to_compare]),
                    alpha=alpha
                )
                print(tukey)
            
            # Para Kruskal-Wallis (Mann-Whitney U com correção Bonferroni)
            else:
                for (model1, data1), (model2, data2) in combinations(zip(models, data_to_compare), 2):
                    _, p_val = mannwhitneyu(data1, data2)
                    print(f"{model1} vs {model2}: p = {p_val:.4f} (Mann-Whitney U)")
    else:
        print(f"Conclusão (α={alpha}): Não há diferenças significativas.")

def main():
    # Configurações
    folder_path = "./CSV/"
    model_prefixes = ['IQC_Linha', 'IQC_Coluna', 'IQC_AIL_Linha', 'IQC_AIL_Coluna', 'IQC_Angle_Linha', 'IQC_Angle_Coluna', 'IQCpQ_NF4_Linha', 'IQCpQ_NF4_Coluna', 'IQCNDsE_Linha', 'IQCNDsE_Coluna']
    target_columns = ['Scores', 'F1_Scores', 'Negativity_Class_0']
    alpha = 0.05

    try:
        model_data = load_and_merge_models(folder_path, model_prefixes, target_columns)
        
        # Filtra modelos com dados
        valid_models = {k: v for k, v in model_data.items() if not v.empty}
        print("Modelos carregados:")
        for model, df in valid_models.items():
            print(f"- {model}: {len(df)} amostras | Colunas: {list(df.columns)}")
        
        # Análise para cada coluna
        for column in target_columns:
            print(f"\n{'='*50}\nAnálise para: {column}\n{'='*50}")
            compare_models(valid_models, column, alpha)
            
    except Exception as e:
        print(f"Erro: {str(e)}")

if __name__ == "__main__":
    main()

Erro: No objects to concatenate


In [7]:
import os
import pandas as pd
import numpy as np
from scipy.stats import shapiro, ttest_ind, mannwhitneyu, f_oneway, levene, kruskal
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from itertools import combinations
from datetime import datetime

def load_model_data(folder_path, model_names):
    """
    Carrega dados de múltiplos modelos a partir de arquivos CSV.
    
    Args:
        folder_path (str): Caminho para a pasta com os arquivos
        model_names (list): Lista de nomes de modelos (ex: ['IQC_Linha', 'IQC_CNN'])
    
    Returns:
        dict: Dicionário com DataFrames para cada modelo
    """
    model_data = {model: [] for model in model_names}
    
    for file in os.listdir(folder_path):
        if file.endswith('.csv'):
            for model in model_names:
                if model in file:  # Verifica se o nome do modelo está no nome do arquivo
                    df = pd.read_csv(os.path.join(folder_path, file))
                    model_data[model].append(df)
                    break
    
    # Concatena todos os dados para cada modelo
    return {model: pd.concat(dfs, ignore_index=True) for model, dfs in model_data.items()}

def check_assumptions(data_dict, target_column, alpha=0.05):
    """
    Verifica normalidade (Shapiro-Wilk) e homocedasticidade (Levene).
    
    Returns:
        tuple: (normality_results, all_normal, homogeneous)
    """
    normality_results = {}
    all_normal = True
    data_to_compare = []
    
    for model, df in data_dict.items():
        if target_column in df.columns:
            data = df[target_column].dropna()
            stat, p = shapiro(data)
            is_normal = p > alpha
            normality_results[model] = {
                'p_value': p,
                'is_normal': is_normal,
                'data': data
            }
            all_normal &= is_normal
            data_to_compare.append(data)
        else:
            print(f"Aviso: Coluna '{target_column}' não encontrada no modelo {model}")
            normality_results[model] = {
                'p_value': np.nan,
                'is_normal': False,
                'data': None
            }
            all_normal = False
    
    # Verifica homocedasticidade apenas se todos forem normais
    homogeneous = False
    if all_normal and len(data_to_compare) >= 2:
        _, p_levene = levene(*data_to_compare)
        homogeneous = p_levene > alpha
    
    return normality_results, all_normal, homogeneous

def compare_models(data_dict, target_column, alpha=0.05):
    """
    Compara modelos usando testes estatísticos apropriados.
    """
    # Verifica suposições
    normality_results, all_normal, homogeneous = check_assumptions(data_dict, target_column, alpha)
    
    # Imprime resultados das verificações
    print("\n=== Verificação de Suposições ===")
    print("Teste de Normalidade (Shapiro-Wilk):")
    for model, result in normality_results.items():
        print(f"{model}: p = {result['p_value']:.4f} | {'Normal' if result['is_normal'] else 'Não normal'}")
    
    if all_normal:
        print(f"\nTeste de Homocedasticidade (Levene): {'Homogêneo' if homogeneous else 'Heterogêneo'}")
    
    # Prepara dados para comparação
    models = [k for k, v in normality_results.items() if v['data'] is not None]
    data_to_compare = [normality_results[model]['data'] for model in models]
    
    # Seleciona teste estatístico
    if len(models) == 2:
        if all_normal:
            stat, p = ttest_ind(*data_to_compare, equal_var=homogeneous)
            test_name = "Teste t de Student (Welch)" if not homogeneous else "Teste t de Student"
        else:
            stat, p = mannwhitneyu(*data_to_compare)
            test_name = "Teste de Mann-Whitney U"
    else:
        if all_normal:
            stat, p = f_oneway(*data_to_compare)
            test_name = "ANOVA unidirecional"
        else:
            stat, p = kruskal(*data_to_compare)
            test_name = "Teste de Kruskal-Wallis"
    
    # Executa e mostra resultados
    print(f"\n=== Resultado do Teste ({test_name}) ===")
    print(f"Estatística = {stat:.4f}, p-value = {p:.4f}")
    
    if p < alpha:
        print(f"\nConclusão (α={alpha}): Há diferenças significativas entre os modelos.")
        
        # Testes post-hoc
        if len(models) > 2:
            print("\n=== Testes Post-Hoc ===")
            
            if all_normal:
                # Tukey HSD para ANOVA
                tukey = pairwise_tukeyhsd(
                    endog=np.concatenate(data_to_compare),
                    groups=np.repeat(models, [len(x) for x in data_to_compare]),
                    alpha=alpha
                )
                print(tukey.summary())
            else:
                # Mann-Whitney U com correção Bonferroni para Kruskal-Wallis
                n_comparisons = len(list(combinations(models, 2)))
                bonferroni_alpha = alpha / n_comparisons
                
                print(f"Comparações par a par (Mann-Whitney U com correção Bonferroni α={bonferroni_alpha:.4f}):")
                for (model1, data1), (model2, data2) in combinations(zip(models, data_to_compare), 2):
                    _, p_val = mannwhitneyu(data1, data2)
                    significant = "***" if p_val < bonferroni_alpha else ""
                    print(f"{model1} vs {model2}: p = {p_val:.4f} {significant}")
    else:
        print(f"\nConclusão (α={alpha}): Não há diferenças significativas entre os modelos.")

def analyze_all_metrics(folder_path, model_names):
    """
    Analisa todas as métricas disponíveis para os modelos especificados.
    """
    try:
        # Carrega os dados
        model_data = load_model_data(folder_path, model_names)
        
        # Verifica modelos carregados
        valid_models = {k: v for k, v in model_data.items() if not v.empty}
        if not valid_models:
            raise ValueError("Nenhum dado válido encontrado para os modelos especificados.")
        
        print("\n=== Modelos Carregados ===")
        for model, df in valid_models.items():
            print(f"\nModelo: {model}")
            print(f"Amostras: {len(df)}")
            print("Colunas disponíveis:", list(df.columns))
        
        # Define as métricas padrão para análise
        target_metrics = ['Accuracy', 'F1_Scores']
        
        # Adiciona colunas de negatividade se existirem
        for col in valid_models[list(valid_models.keys())[0]].columns:
            if 'Negativity_Class_' in col:
                target_metrics.append(col)
        
        # Realiza análise para cada métrica
        for metric in target_metrics:
            print(f"\n{'='*50}\nAnálise para a métrica: {metric}\n{'='*50}")
            compare_models(valid_models, metric)
            
    except Exception as e:
        print(f"\nErro durante a análise: {str(e)}")

if __name__ == "__main__":
    # Configurações
    folder_path = "./CSV/"
    model_prefixes = ['IQC_Linha', 'IQC_Coluna', 'IQC_AIL_Linha', 'IQC_AIL_Coluna', 'IQC_Angle_Linha', 'IQC_Angle_Coluna', 'IQCpQ_NF4_Linha', 'IQCpQ_NF4_Coluna', 'IQCNDsE_Linha', 'IQCNDsE_Coluna']
    target_columns = ['Scores', 'F1_Scores', 'Negativity_Class_0']
    alpha = 0.05
    # Executa a análise completa
    analyze_all_metrics(folder_path, model_prefixes)


Erro durante a análise: No objects to concatenate


In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.stats import shapiro, ttest_ind, f_oneway, kruskal
from collections import defaultdict

In [ ]:
# Caminho da pasta com arquivos .csv
pasta = "CAMINHO/DA/PASTA/COM/CSV"

# Carregar todos os arquivos CSV
dados_modelos = defaultdict(list)

for arquivo in os.listdir(pasta):
    if arquivo.endswith(".csv"):
        caminho = os.path.join(pasta, arquivo)
        df = pd.read_csv(caminho)
        
        # Extrair nome do modelo do nome do arquivo (assume prefixo antes de "_metrics_")
        nome_modelo = arquivo.split("_metrics_")[0]
        dados_modelos[nome_modelo].append(df)

# Concatenar os dados por modelo
modelos_dfs = {modelo: pd.concat(lista, ignore_index=True) for modelo, lista in dados_modelos.items()}

# Comparar por coluna
colunas = ["Scores", "F1_Scores"]  # Adapte se necessário
resultados = {}

for coluna in colunas:
    print(f"\n--- Análise da métrica: {coluna} ---")
    amostras = []
    normalidades = {}
    
    for modelo, df in modelos_dfs.items():
        if coluna not in df.columns:
            continue
        dados = df[coluna].dropna().values
        amostras.append((modelo, dados))
        
        # Teste de normalidade de Shapiro-Wilk
        stat, p_shapiro = shapiro(dados)
        normalidades[modelo] = p_shapiro > 0.05  # True se normal
        print(f"{modelo}: Shapiro-Wilk p = {p_shapiro:.4f} → {'Normal' if p_shapiro > 0.05 else 'Não normal'}")

    # Separar os valores para teste estatístico
    nomes_modelos = [m for m, _ in amostras]
    valores_modelos = [v for _, v in amostras]

    if len(amostras) < 2:
        print("⚠️ Dados insuficientes para comparar modelos.")
        continue

    # Verifica se todas as distribuições são normais
    if all(normalidades.values()):
        if len(amostras) == 2:
            stat, p_ttest = ttest_ind(valores_modelos[0], valores_modelos[1], equal_var=False)
            print(f"\nTeste t de Student entre {nomes_modelos[0]} e {nomes_modelos[1]}:")
            print(f"t = {stat:.4f}, p = {p_ttest:.4f} → {'Diferença significativa' if p_ttest < 0.05 else 'Sem diferença'}")
            resultados[coluna] = ('ttest', p_ttest)
        else:
            stat, p_anova = f_oneway(*valores_modelos)
            print("\nANOVA:")
            print(f"F = {stat:.4f}, p = {p_anova:.4f} → {'Diferença significativa' if p_anova < 0.05 else 'Sem diferença'}")
            resultados[coluna] = ('anova', p_anova)
    else:
        stat, p_kw = kruskal(*valores_modelos)
        print("\nTeste de Kruskal-Wallis:")
        print(f"H = {stat:.4f}, p = {p_kw:.4f} → {'Diferença significativa' if p_kw < 0.05 else 'Sem diferença'}")
        resultados[coluna] = ('kruskal', p_kw)


In [8]:
import os
import pandas as pd
import numpy as np
from scipy.stats import shapiro, ttest_ind, mannwhitneyu, kruskal
from collections import defaultdict


In [10]:

# Prefixos dos modelos que queremos considerar
model_prefixes = [
    'IQC_Linha', 'IQC_Coluna', 'IQC_AIL_Linha', 'IQC_AIL_Coluna',
    'IQC_Angle_Linha', 'IQC_Angle_Coluna', 'IQCpQ_NF4_Linha', 'IQCpQ_NF4_Coluna',
    'IQCNDsE_Linha', 'IQCNDsE_Coluna'
]

def load_metrics_from_csv(directory):
    model_data = defaultdict(list)

    for filename in os.listdir(directory):
        if filename.endswith(".csv"):
            for prefix in model_prefixes:
                if filename.startswith(prefix):
                    filepath = os.path.join(directory, filename)
                    df = pd.read_csv(filepath)
                    model_data[prefix].append(df)
                    break  # para não associar o mesmo arquivo a múltiplos prefixos

    # Concatena os DataFrames de cada modelo
    for model in model_data:
        model_data[model] = pd.concat(model_data[model], ignore_index=True)

    return model_data

def test_normality_and_compare(models_data, metric='F1_Scores'):
    normalities = {}
    
    print(f"\n--- Teste de Normalidade (Shapiro-Wilk) para '{metric}' ---")
    for model, df in models_data.items():
        if metric not in df.columns:
            print(f"{model}: métrica '{metric}' não encontrada.")
            continue
        stat, p = shapiro(df[metric])
        normal = p > 0.05
        normalities[model] = normal
        print(f"{model}: p-value = {p:.4f} → {'Normal' if normal else 'Não Normal'}")

    models = [model for model in models_data if metric in models_data[model].columns]
    data = [models_data[model][metric].values for model in models]

    print(f"\n--- Comparação Estatística para '{metric}' ---")
    if len(models) == 2:
        a, b = data
        if normalities[models[0]] and normalities[models[1]]:
            stat, p = ttest_ind(a, b, equal_var=False)
            print(f"Teste t-Student entre {models[0]} e {models[1]}: p-value = {p:.4f}")
        else:
            stat, p = mannwhitneyu(a, b, alternative='two-sided')
            print(f"Teste Mann-Whitney entre {models[0]} e {models[1]}: p-value = {p:.4f}")
    elif len(models) > 2:
        stat, p = kruskal(*data)
        print(f"Teste Kruskal-Wallis entre {len(models)} modelos: p-value = {p:.4f}")
    else:
        print("Número insuficiente de modelos com dados válidos para comparação.")

# -------------------------------
# USO
# -------------------------------

# Caminho da pasta onde estão os .csv
directory = "./CSV/"

# Carrega os dados por modelo
models_data = load_metrics_from_csv(directory)

# Realiza os testes para uma métrica
test_normality_and_compare(models_data, metric='F1_Scores')
test_normality_and_compare(models_data, metric='Accuracy')



--- Teste de Normalidade (Shapiro-Wilk) para 'F1_Scores' ---

--- Comparação Estatística para 'F1_Scores' ---
Número insuficiente de modelos com dados válidos para comparação.

--- Teste de Normalidade (Shapiro-Wilk) para 'Accuracy' ---

--- Comparação Estatística para 'Accuracy' ---
Número insuficiente de modelos com dados válidos para comparação.
